# CA2, Phase 3: roaming and the sticky client

This notebook is **optional**: any language or tool is fine for the analysis (see the setup section of the assignment). It only **loads** simulation results; the models, the state machine, the analysis and the plots are yours to write, in new cells under each section heading.

Rules:
- Your group's parameters come from `../params.json`, including `alpha` for the mean power curve.
- Every number and plot in your report must be reproducible from the code you hand in; the TAs do not run it, but at the in-person delivery you will rerun parts of it and change them.

In [ ]:
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from omnetpp.scave import results

with open(os.path.join("..", "params.json")) as f:
    P = json.load(f)
D, V, BI, N_CH = P["D"], P["v"], P["BI"] / 1000, P["nCh"]   # m, m/s, s, channels scanned
ALPHA, SIGMA = P["alpha"], P["sigma"]                        # true values, to check your fit
T_WALK = D / V                                               # s: after this the walker stands next to ap2
print(f"D = {D} m, v = {V} m/s, BI = {BI * 1000:.0f} ms, nCh = {N_CH}, walk lasts {T_WALK:.1f} s")

## Loading results

`load_vectors(config)` returns every vector of one config, one row per run and vector, with `repetition` as a column.
`vector(vecs, rep, module, name)` returns the times and values of one vector in run `rep` (two empty arrays if that run did not record it).

Vectors you will need (full module paths start with `Roaming.`):

| config | module | vector | one sample per |
|---|---|---|---|
| `Survey` | `meter.wlan[0].radio`, `meter.wlan[1].radio` | `minSnir:vector` | beacon of ap1 / ap2 the meter received (SNIR as a ratio, not dB) |
| `Walk` | `ap1.sink`, `ap2.sink` | `packets:vector(packetLength)` | voice frame delivered by that AP |
| `Walk` | `host.wlan.agent` | `sentRequest:vector`, `acceptConfirm:vector`, `dropConfirm:vector` | request to / answer from the management module (the value is a code) |
| `Walk` | `host.wlan.radio` | `radioChannel:vector` | channel change of the station's radio |
| `Walk` | `host.wlan.radio` | `minSnir:vector` | frame the station received |
| `Walk` | `host.wlan.mac.dcf` | `packetSentToPeerWithoutRetry:vector(packetBytes)`, `packetSentToPeerWithRetry:vector(packetBytes)` | first transmission / retransmission of a frame |

In [ ]:
RESULTS = "results"


def load_vectors(config):
    df = results.read_result_files(os.path.join(RESULTS, f"{config}-*.vec"))
    vecs = results.get_vectors(df, include_itervars=True, include_runattrs=True)
    vecs["repetition"] = pd.to_numeric(vecs["repetition"])
    return vecs


def vector(vecs, rep, module, name):
    row = vecs[(vecs.repetition == rep) & (vecs.module == "Roaming." + module) & (vecs.name == name)]
    if row.empty:
        return np.array([]), np.array([])
    return row.iloc[0].vectime, row.iloc[0].vecvalue

Example: the first beacons of ap1 that the meter received in run 0 of `Survey` (run that config first).

In [ ]:
survey = load_vectors("Survey")
t, snir = vector(survey, 0, "meter.wlan[0].radio", "minSnir:vector")
pd.DataFrame({"t (s)": t[:5], "SNIR": snir[:5]})

## 3-1  Anatomy of a handover

## 3-2  The sticky client

## 3-3  Your roaming algorithm